In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('factory_sensor_simulator_2040.csv')
eps = 1e-5

In [ ]:
df['Specific_Vibration_Energy'] = (df['Vibration_mms']**2) / (np.log(df['Power_Consumption_kW'] + 1) + eps)
df['Cumulative_Mechanical_Stress'] = df['Operational_Hours'] * np.sqrt(df['Power_Consumption_kW'] * df['Temperature_C'])
type_temp_mean = df.groupby('Machine_Type')['Temperature_C'].transform('mean')
df['Thermal_Degradation_Index'] = np.exp(df['Temperature_C'] / (type_temp_mean + eps))

In [ ]:
df['Maintenance_Decay_Factor'] = 1 - np.exp(-df['Last_Maintenance_Days_Ago'] / 90.0)
df['MTBF_Proxy'] = df['Operational_Hours'] / (df['Failure_History_Count'] + 1)
df['Maintenance_Inefficiency_Index'] = (df['Failure_History_Count'] + 1) / (df['Maintenance_History_Count'] + 1)

In [ ]:
def assign_cohort(year):
    if year < 2000:
        return 'Legacy_Pre2000'
    elif 2000 <= year <= 2022:
        return 'Modern_Standard'
    else:
        return 'NextGen_FutureTech'
df['Machine_Tech_Cohort'] = df['Installation_Year'].apply(assign_cohort)
#temp

In [ ]:
sensor_cols = ['Temperature_C', 'Vibration_mms', 'Sound_dB', 'Power_Consumption_kW']
for col in sensor_cols:
    mean = df.groupby('Machine_Type')[col].transform('mean')
    std = df.groupby('Machine_Type')[col].transform('std') + eps
    df[f'{col}_Z'] = (df[col] - mean) / std
df['Multivariate_Anomaly_Score'] = np.sqrt(
    df['Temperature_C_Z']**2 + 
    df['Vibration_mms_Z']**2 + 
    df['Sound_dB_Z']**2 + 
    df['Power_Consumption_kW_Z']**2
)
df.drop(columns=[f'{col}_Z' for col in sensor_cols], inplace=True)
print("Multivariate anomaly score calculated.")

In [ ]:
df['Flag_Critical_Fluid'] = np.where((df['Oil_Level_pct'] < df['Oil_Level_pct'].quantile(0.25)) | (df['Coolant_Level_pct'] < df['Coolant_Level_pct'].quantile(0.25)), 1, 0)
df['Flag_High_Stress'] = np.where((df['Temperature_C'] > df['Temperature_C'].quantile(0.75)) | (df['Vibration_mms'] > df['Vibration_mms'].quantile(0.75)), 1, 0)
df['Flag_Overdue_Maintenance'] = np.where(df['Last_Maintenance_Days_Ago'] > df['Last_Maintenance_Days_Ago'].quantile(0.75), 1, 0)
print("Binary flags generated successfully.")

In [ ]:
print(f"Final Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
new_engineered_cols = [
    'Specific_Vibration_Energy', 
    'Cumulative_Mechanical_Stress', 
    'Thermal_Degradation_Index', 
    'Maintenance_Decay_Factor', 
    'MTBF_Proxy', 
    'Maintenance_Inefficiency_Index', 
    'Multivariate_Anomaly_Score'
]
df[new_engineered_cols]